In [ ]:
!pip install yfinance

In [7]:
import yfinance as yf
import pandas as pd
from pymongo import MongoClient, UpdateOne
import sys
import akshare as ak

In [23]:
import os
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi
uri = os.environ["MONGODB_URI"]

client = MongoClient(uri, server_api=ServerApi('1'))

try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

Pinged your deployment. You successfully connected to MongoDB!


In [19]:
try:
    db = client["test"]
    print("Connection successful!")
    print("Collections:", db.list_collection_names())
except Exception as e:
    print("Connection failed:", e)

Connection successful!
Collections: []


In [24]:
db = client["sse_stock_db"]        # or any database name you want
collection = db["daily_data"]      # your main stock data collection
meta = db["fetch_status"]          # to track completed stocks

In [86]:
done_tickers = set(collection.distinct("ticker"))
print(f"Already in DB: {len(done_tickers)} tickers")

Already in DB: 56 tickers


In [91]:
stocks = ak.stock_info_sh_name_code()
codes = stocks["证券代码"].astype(str).tolist()

print(f"Total SSE stocks: {len(codes)}")

Total SSE stocks: 1700


In [79]:
print(codes[62:98])

['600088', '600089', '600094', '600095', '600096', '600097', '600098', '600099', '600100', '600101', '600103', '600104', '600105', '600106', '600107', '600108', '600109', '600110', '600111', '600113', '600114', '600115', '600116', '600117', '600118', '600119', '600120', '600121', '600123', '600125', '600126', '600127', '600128', '600129', '600130', '600131']


In [81]:
is_complete("600096.SS")

False

In [92]:
import yfinance as yf
import pandas as pd
from pymongo import UpdateOne
import sys

def log(msg):
    print(msg)
    sys.stdout.flush()

def is_complete(ticker):
    status = meta.find_one({"ticker": ticker})
    return status is not None and status.get("completed", False)

def fetch_one(code):
    ticker = code + ".SS"

    if is_complete(ticker):
        log(f"⏭️ Skipping {code}, already completed")
        return

    try:
        log(f"Fetching {code} ...")
        df = yf.download(ticker, start="2010-01-01", progress=False)
        if df.empty:
            log(f"⚠️ No data for {code}")
            return

        # Reset index to get 'Date' as column
        df = df.reset_index()

        # Fix MultiIndex column if exists
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]

        # Rename to standard columns
        df.rename(columns={
            "Date": "date",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Adj Close": "adj_close",
            "Volume": "volume"
        }, inplace=True)

        # Ensure ticker column exists **before to_dict**
        df["ticker"] = ticker

        # Optional: check first 2 rows
        print(df.head(2))

        # Build bulk ops
        ops = [
            UpdateOne(
                {"ticker": r["ticker"], "date": r["date"]},
                {"$set": r},
                upsert=True
            )
            for r in df.to_dict("records")
        ]

        if ops:
            collection.bulk_write(ops, ordered=False)
            log(f"  ✅ Inserted {len(ops)} records for {code}")

        # Mark completion
        meta.update_one(
            {"ticker": ticker},
            {"$set": {"completed": True, "rows": len(df)}},
            upsert=True
        )

    except Exception as e:
        log(f"❌ Failed {code}: {e}")

In [61]:
completed = set(
    m["ticker"]
    for m in meta.find({"completed": True}, {"ticker": 1})
)

In [63]:
def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]
completed = set(
    m["ticker"] for m in meta.find({"completed": True}, {"ticker": 1})
)


In [93]:
# run sequentially
for code in codes:
    fetch_one(code)

⏭️ Skipping 600000, already completed
⏭️ Skipping 600004, already completed
⏭️ Skipping 600006, already completed
⏭️ Skipping 600007, already completed
⏭️ Skipping 600008, already completed
⏭️ Skipping 600009, already completed
⏭️ Skipping 600010, already completed
⏭️ Skipping 600011, already completed
⏭️ Skipping 600012, already completed
⏭️ Skipping 600015, already completed
⏭️ Skipping 600016, already completed
⏭️ Skipping 600017, already completed
⏭️ Skipping 600018, already completed
⏭️ Skipping 600019, already completed
⏭️ Skipping 600020, already completed
⏭️ Skipping 600021, already completed
⏭️ Skipping 600022, already completed
⏭️ Skipping 600023, already completed
⏭️ Skipping 600025, already completed
⏭️ Skipping 600026, already completed
⏭️ Skipping 600027, already completed
⏭️ Skipping 600028, already completed
⏭️ Skipping 600029, already completed
⏭️ Skipping 600030, already completed
⏭️ Skipping 600031, already completed
⏭️ Skipping 600032, already completed
⏭️ Skipping 

KeyboardInterrupt: 

In [94]:
done_tickers = set(collection.distinct("ticker"))
print(f"Already in DB: {len(done_tickers)} tickers")

Already in DB: 797 tickers
